# 02 — Spatial CNN Training

Trains the custom 5-block CNN on 224×224 RGB face crops from FF++ only.

**Environment:** Colab T4 (15 GB VRAM) or A100  
**Batch size:** 64 on A100 | 32 on T4  
**Checkpointing:** every epoch → Google Drive  
**Mixed precision:** `torch.cuda.amp` (required)

In [ ]:
# ── Colab setup (run every session — VMs reset between sessions) ─────────────
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install -q opencv-python mediapipe tqdm scikit-learn pandas

# DRIVE_ROOT = '/content/drive/MyDrive/deepfake-detection'
# DATA_ROOT  = f'{DRIVE_ROOT}/data'
# CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints/spatial'
# import os; os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from models.spatial.spatial_cnn import SpatialCNN

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

BATCH_SIZE = 64 if device == 'cuda' else 4  # 32 on T4, 64 on A100
EPOCHS     = 20
LR         = 1e-4
CKPT_DIR   = '../results'  # override with Drive path on Colab

model  = SpatialCNN(dropout=0.5).to(device)
scaler = GradScaler(enabled=(device == 'cuda'))
optim  = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
print(f'Batch size: {BATCH_SIZE}  |  Epochs: {EPOCHS}')

In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
import os

def save_checkpoint(model, optimizer, epoch, val_auc, path):
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_auc':   val_auc,
    }, path)
    print(f'  Checkpoint saved → {path}')

def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f'  Resumed from epoch {ckpt["epoch"]}  val_auc={ckpt["val_auc"]:.4f}')
    return ckpt['epoch']

CKPT_PATH = os.path.join(CKPT_DIR, 'spatial_latest.pt')
start_epoch = 0
if os.path.exists(CKPT_PATH):
    start_epoch = load_checkpoint(model, optim, CKPT_PATH)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
# TODO: implement SpatialDataset(csv_path, split) that:
#   - reads frame_path and label from CSV
#   - applies torchvision transforms (RandomHorizontalFlip, ColorJitter, Normalize)
#   - returns (image_tensor, label_float)
# 
# from torch.utils.data import DataLoader
# train_loader = DataLoader(SpatialDataset(train_csv, 'train'), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
# val_loader   = DataLoader(SpatialDataset(val_csv,   'val'),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
pass

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
import numpy as np

def train_epoch(model, loader, optimizer, loss_fn, scaler, device):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast(enabled=(device == 'cuda')):
            logits = model(imgs)
            loss   = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with autocast(enabled=(device == 'cuda')):
            probs = torch.sigmoid(model(imgs)).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    auc = roc_auc_score(all_labels, all_probs)
    acc = ((np.array(all_probs) > 0.5) == np.array(all_labels)).mean()
    return auc, acc

# TODO: uncomment once DataLoaders are ready
# for epoch in range(start_epoch, EPOCHS):
#     train_loss       = train_epoch(model, train_loader, optim, loss_fn, scaler, device)
#     val_auc, val_acc = evaluate(model, val_loader, device)
#     print(f'Epoch {epoch+1}/{EPOCHS}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  val_acc={val_acc:.4f}')
#     save_checkpoint(model, optim, epoch+1, val_auc, CKPT_PATH)

In [ ]:
# ── Evaluation: AUC per manipulation type ────────────────────────────────────
# TODO: load test split, group by manipulation_type, compute AUC per group
# Save results with: from utils.results_logger import log_results
# log_results('spatial_cnn', metrics_dict)
pass